# PromptForge Phase 2 — Prompt Optimizer (LoRA)

Self-contained Colab notebook (same style as Phase 1).

**Runtime → GPU** (T4 / L4 / A100)

Pipeline:
```text
Original prompt + quality analysis
        ↓
  LoRA fine-tuned LLM
        ↓
  Optimized prompt
```

Base model default: `Qwen/Qwen2.5-0.5B-Instruct` (Colab-friendly).


In [ ]:
!pip install -q \
    transformers \
    datasets \
    accelerate \
    peft \
    scikit-learn \
    pandas \
    numpy \
    scipy \
    huggingface_hub \
    evaluate \
    pyyaml


In [ ]:
import os
import json
import random

import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorForLanguageModeling,
)

from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
)

from sklearn.model_selection import train_test_split

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Enable a Colab GPU runtime before training.")


In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_SEQ_LENGTH = 1024
MAX_NEW_TOKENS = 512
NUM_EXAMPLES = 10000

OUTPUT_DIR = "./promptforge-optimizer"
FINAL_MODEL_DIR = "/content/promptforge-optimizer-model"
DATASET_PATH = "/content/promptforge_optimizer_dataset.csv"

print("Base model:", BASE_MODEL)
print("Examples:", NUM_EXAMPLES)


## Dataset generation (weak prompt → optimized prompt)


In [ ]:
TASKS = {
    "coding": [
        "Build a REST API",
        "Create a React application",
        "Write a Python script",
        "Build a CLI tool",
        "Create a database schema",
    ],
    "writing": [
        "Write a blog post",
        "Write an email",
        "Write a technical article",
        "Create a product description",
        "Write a LinkedIn post",
    ],
    "research": [
        "Research a technology",
        "Compare two databases",
        "Analyze a software architecture",
        "Explain a machine learning technique",
        "Evaluate a programming language",
    ],
    "data": [
        "Analyze a dataset",
        "Create a data visualization",
        "Build a machine learning model",
        "Clean a dataset",
        "Generate a statistical report",
    ],
    "creative": [
        "Create a story",
        "Write a game concept",
        "Design a character",
        "Create a marketing campaign",
        "Generate a product idea",
    ],
}

AUDIENCES = [
    "beginner developers",
    "experienced developers",
    "software engineers",
    "students",
    "technical managers",
    "startup founders",
    "data scientists",
    "general users",
]

CONSTRAINTS = [
    "Use Python 3.12.",
    "Use TypeScript and React.",
    "Keep the solution under 200 lines.",
    "Return the answer as Markdown.",
    "Include complete working code.",
    "Do not use external libraries.",
    "Include error handling.",
    "Make the solution production-ready.",
    "Use PostgreSQL.",
    "Make it mobile responsive.",
]

OUTPUT_FORMATS = [
    "Return the answer as a numbered list.",
    "Return complete source code.",
    "Return JSON.",
    "Return a step-by-step explanation.",
    "Return a Markdown document.",
    "Include examples.",
]

CONTEXTS = [
    "This is for a university project.",
    "This is for a production SaaS application.",
    "This is for an internal developer tool.",
    "This will be used by beginners.",
    "This is a prototype for a startup.",
    "This will run locally on a laptop.",
]

LEVEL0 = [
    "Make an app.",
    "Build something.",
    "Write something good.",
    "Help me with this.",
    "Create a website.",
    "Make this better.",
    "Analyze this.",
    "Build me a project.",
]

SYSTEM_PROMPT = (
    "You are PromptForge Optimizer. Rewrite the user's prompt into a clear, "
    "specific, complete, and actionable LLM prompt. Preserve the original intent. "
    "Add missing audience, context, constraints, and output format when needed. "
    "Return ONLY the optimized prompt text."
)


In [ ]:
def clamp(value):
    return max(0, min(100, int(round(value))))


def scores_for_level(level):
    clarity = clamp(30 + level * 17 + random.randint(-5, 5))
    specificity = clamp(15 + level * 20 + random.randint(-5, 5))
    context = clamp(10 + level * 20 + random.randint(-5, 5))
    goal_definition = clamp(25 + level * 17 + random.randint(-5, 5))
    constraints = clamp(5 + level * 22 + random.randint(-5, 5))
    completeness = clamp(15 + level * 20 + random.randint(-5, 5))
    actionability = clamp(20 + level * 18 + random.randint(-5, 5))
    dims = {
        "clarity": clarity,
        "specificity": specificity,
        "context": context,
        "goal_definition": goal_definition,
        "constraints": constraints,
        "completeness": completeness,
        "actionability": actionability,
    }
    dims["quality_score"] = clamp(float(np.mean(list(dims.values()))))
    return dims


def weak_prompt(domain, task, level):
    if level <= 0:
        return random.choice(LEVEL0)
    if level == 1:
        return f"{task}."
    if level == 2:
        return f"{task} for {random.choice(AUDIENCES)}."
    if level == 3:
        return (
            f"{task} for {random.choice(AUDIENCES)}. "
            f"{random.choice(CONTEXTS)} "
            f"{random.choice(OUTPUT_FORMATS)}"
        )
    return (
        f"{task} for {random.choice(AUDIENCES)}. "
        f"{random.choice(CONTEXTS)} "
        f"Use reasonable defaults."
    )


def build_optimized(domain, task):
    task_detail = task[0].lower() + task[1:]
    audience = random.choice(AUDIENCES)
    context = random.choice(CONTEXTS)
    c1 = random.choice(CONSTRAINTS)
    c2 = random.choice(CONSTRAINTS)
    output = random.choice(OUTPUT_FORMATS)

    if domain == "coding":
        return (
            f"Build a production-ready {task_detail} for {audience}.\n"
            f"{context}\n\n"
            f"Requirements:\n"
            f"- {c1}\n"
            f"- {c2}\n"
            f"- Include error handling and basic tests\n\n"
            f"Return:\n"
            f"1. Project structure\n"
            f"2. Complete implementation\n"
            f"3. Setup / run instructions\n"
            f"4. Example usage"
        )
    if domain == "writing":
        return (
            f"Write a high-quality {task_detail} for {audience}.\n"
            f"{context}\n\n"
            f"Requirements:\n"
            f"- Clear structure with headings\n"
            f"- Concrete examples\n"
            f"- {output}\n\n"
            f"Tone: professional, concise, and useful."
        )
    return (
        f"Complete this task thoroughly: {task_detail}.\n"
        f"Audience: {audience}.\n"
        f"{context}\n\n"
        f"Constraints:\n- {c1}\n- {c2}\n"
        f"{output}"
    )


def missing_from_scores(scores):
    missing = []
    if scores["context"] < 45:
        missing.append("context")
    if scores["goal_definition"] < 45:
        missing.append("goal")
    if scores["constraints"] < 45:
        missing.append("constraints")
    if scores["specificity"] < 45:
        missing.append("specific_requirements")
    if scores["completeness"] < 45:
        missing.append("output_format")
    return missing


def issues_from_scores(scores):
    issues = []
    if scores["specificity"] < 40:
        issues.append("too_vague")
    if scores["context"] < 40:
        issues.append("missing_context")
    if scores["goal_definition"] < 45:
        issues.append("ambiguous_objective")
    if scores["constraints"] < 40:
        issues.append("insufficient_constraints")
    return issues


def format_user(prompt, analysis, task_type):
    dims = analysis["dimensions"]
    lines = [
        f"Task type: {task_type}",
        "Original prompt:",
        prompt.strip(),
        "",
        "Quality analysis:",
        f"- quality_score: {analysis['quality_score']}",
    ]
    for key, value in dims.items():
        lines.append(f"- {key}: {value}")
    if analysis["issues"]:
        lines.append(f"- issues: {', '.join(analysis['issues'])}")
    if analysis["missing_information"]:
        lines.append(
            f"- missing_information: {', '.join(analysis['missing_information'])}"
        )
    lines.extend([
        "",
        "Rewrite this into a high-quality optimized prompt.",
        "Return only the optimized prompt.",
    ])
    return "\n".join(lines)


def build_training_text(prompt, analysis, optimized, task_type):
    user = format_user(prompt, analysis, task_type)
    return (
        f"<|system|>\n{SYSTEM_PROMPT}\n"
        f"<|user|>\n{user}\n"
        f"<|assistant|>\n{optimized.strip()}"
    )


def generate_example():
    domain = random.choice(list(TASKS.keys()))
    task = random.choice(TASKS[domain])
    level = random.choices(
        [0, 1, 2, 3, 4],
        weights=[0.25, 0.25, 0.25, 0.15, 0.10],
        k=1,
    )[0]

    prompt = weak_prompt(domain, task, level)
    scores = scores_for_level(level)
    analysis = {
        "quality_score": scores["quality_score"],
        "dimensions": {k: v for k, v in scores.items() if k != "quality_score"},
        "issues": issues_from_scores(scores),
        "missing_information": missing_from_scores(scores),
    }
    optimized = build_optimized(domain, task)

    return {
        "prompt": prompt,
        "task_type": domain,
        "quality_level": level,
        "quality_score": scores["quality_score"],
        "analysis_json": json.dumps(analysis),
        "optimized_prompt": optimized,
        "training_text": build_training_text(
            prompt, analysis, optimized, domain
        ),
    }


In [ ]:
examples = [generate_example() for _ in range(NUM_EXAMPLES)]
df = pd.DataFrame(examples)

print("Dataset size:", len(df))
print(df.head())
print()
print(df["task_type"].value_counts())
print()
print(df["quality_score"].describe())


In [ ]:
df.to_csv(DATASET_PATH, index=False)
print("Saved:", DATASET_PATH)


In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.10, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "validation": Dataset.from_pandas(val_df.reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df.reset_index(drop=True)),
})
dataset


## Tokenizer + base model + LoRA


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"
print("Tokenizer loaded.")


In [ ]:
def tokenize_function(batch):
    tokenized = tokenizer(
        batch["training_text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )
    tokenized["labels"] = [ids[:] for ids in tokenized["input_ids"]]
    return tokenized


tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

tokenized_dataset


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.config.use_cache = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## Train


In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    gradient_checkpointing=True,
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)


In [ ]:
train_result = trainer.train()
train_result


In [ ]:
evaluation = trainer.evaluate()
evaluation


## Inference helpers


In [ ]:
def optimize_prompt(prompt, analysis=None, task_type="general", model=model, tokenizer=tokenizer):
    model.eval()

    if analysis is None:
        analysis = {
            "quality_score": "unknown",
            "dimensions": {},
            "issues": ["too_vague"],
            "missing_information": ["context", "constraints", "output_format"],
        }

    user = format_user(prompt, analysis, task_type)
    text = (
        f"<|system|>\n{SYSTEM_PROMPT}\n"
        f"<|user|>\n{user}\n"
        f"<|assistant|>\n"
    )

    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[-1]:]
    optimized = tokenizer.decode(generated, skip_special_tokens=True).strip()

    return {
        "original_prompt": prompt,
        "optimized_prompt": optimized,
        "analysis": analysis,
        "task_type": task_type,
    }


In [ ]:
test_prompts = [
    "Make an app.",
    "Build me a website.",
    "Write something about AI.",
    "Make a Python API for beginners.",
]

for prompt in test_prompts:
    # Synthetic weak analysis (in full pipeline, use Phase-1 scorer)
    analysis = {
        "quality_score": 22,
        "dimensions": {
            "clarity": 30,
            "specificity": 15,
            "context": 10,
            "goal_definition": 25,
            "constraints": 8,
            "completeness": 15,
            "actionability": 20,
        },
        "issues": ["too_vague", "missing_context", "insufficient_constraints"],
        "missing_information": ["context", "constraints", "output_format", "goal"],
    }

    result = optimize_prompt(prompt, analysis=analysis, task_type="coding")

    print("=" * 80)
    print("ORIGINAL:")
    print(prompt)
    print("\nOPTIMIZED:")
    print(result["optimized_prompt"])


## Save model (LoRA adapter + tokenizer)


In [ ]:
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

meta = {
    "base_model_name": BASE_MODEL,
    "lora": {
        "r": 16,
        "alpha": 32,
        "dropout": 0.05,
    },
    "max_seq_length": MAX_SEQ_LENGTH,
    "max_new_tokens": MAX_NEW_TOKENS,
}

with open(os.path.join(FINAL_MODEL_DIR, "optimizer_config.json"), "w") as f:
    json.dump(meta, f, indent=2)

print("Model saved to:", FINAL_MODEL_DIR)


## Optional — upload to Hugging Face

```python
from huggingface_hub import login, HfApi
login()
api = HfApi()
api.create_repo("YOUR_USER/PromptForge-Optimizer", exist_ok=True, repo_type="model")
api.upload_folder(
    folder_path="/content/promptforge-optimizer-model",
    repo_id="YOUR_USER/PromptForge-Optimizer",
    repo_type="model",
)
```

## Local usage after download

```bash
pip install -e .
promptforge optimize "Build me a website" --model outputs/promptforge-optimizer-model
```
